# Bài 25: Trích xuất giá trị ảnh theo vị trí (GEE)

## 25.1. Mục tiêu học tập

Sau bài này bạn có thể:

- Trích xuất giá trị ảnh cho một hoặc nhiều điểm
- Trích xuất giá trị cho các vùng khác nhau
- Chuyển kết quả về pandas DataFrame và export ra file excel.

In [ ]:
import ee
import geemap
import geopandas as gpd
import pandas as pd

ee.Initialize()

## 25.2. Chuẩn bị dữ liệu Sentinel-2

Trong mục này, chúng ta sẽ trích xuất giá trị ảnh Sentinel-2 dựa trên dữ liệu điểm và polygons.

In [2]:
# Ví dụ vùng nghiên cứu là Singapore 
roi = gpd.read_file('https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_SGP_1.json')
# Tạo thêm points vs crs=4326 
points = gpd.GeoDataFrame(geometry=[i.geometry.centroid for i in roi.itertuples()], crs='EPSG:4326')
start_date = '2023-06-01'
end_date = '2023-09-30'
# Chuyển đổi roi sang định dạng ee.Geometry
roi = geemap.geopandas_to_ee(roi) # Chuyển đổi GeoDataFrame sang ee.FeatureCollection
points = geemap.geopandas_to_ee(points) # Chuyển đổi GeoDataFrame sang ee.FeatureCollection

sen2col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(roi)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 25))
            .select(['B2', 'B3', 'B4', 'B8'])
          )
print('Số lượng ảnh Sentinel-2:', sen2col.size().getInfo())

Số lượng ảnh Sentinel-2: 2


In [6]:
# Format giá trị trích xuất thành DataFrame
def format_values(values):
    features = values['features']
    data = []
    for feature in features:
        properties = feature['properties']
        data.append(properties)
    return pd.DataFrame(data)

## 25.3. Trích xuất giá trị ảnh với dữ liệu điểm

### 25.3.1. Trích xuất giá trị với một ảnh

In [7]:
# Chọn một ảnh Sentinel-2 đầu tiên để trích xuất giá trị
image = sen2col.first()
# Trích xuất giá trị pixel tại các điểm
values = image.sampleRegions(
            collection=ee.FeatureCollection(points), scale=10, geometries=True
        ).getInfo()
df = format_values(values)
df.head()

,B2,B3,B4,B8
0,794,872,848,1160
1,833,928,730,3280
2,231,450,201,3086
3,612,936,800,3176
4,1310,1518,1620,2082


### 25.3.2. Trích xuất giá trị ảnh theo thời gian

In [20]:
from datetime import datetime
values = sen2col.getRegion(geometry=points, scale=10).getInfo()
df = pd.DataFrame(values[1:], columns=values[0])
df["time"] = [
        datetime.fromtimestamp(timestamp_ms / 1000) for timestamp_ms in df["time"]
    ]
df.head()

,id,longitude,latitude,time,B2,B3,B4,B8
0,20230724T031529_20230724T033402_T48NUG,103.792920,1.415071,2023-07-24 05:37:53.217,231,450,201,3086
1,20230927T031521_20230927T033258_T48NUG,103.792920,1.415071,2023-09-27 05:37:49.482,209,428,213,3220
2,20230724T031529_20230724T033402_T48NUG,103.703538,1.330180,2023-07-24 05:37:53.217,963,1194,1230,2286
3,20230927T031521_20230927T033258_T48NUG,103.703538,1.330180,2023-09-27 05:37:49.482,1562,1770,1852,2684
4,20230724T031529_20230724T033402_T48NUG,103.831907,1.306106,2023-07-24 05:37:53.217,794,872,848,1160


## 25.4. Trích xuất giá trị ảnh với đối tượng đa giác

### 25.4.1. Trích xuất giá trị với một ảnh 

In [ ]:
# Trích xuất giá trị trung bình của ảnh đầu tiên cho toàn bộ roi
results = sen2col.first().reduceRegions(
            collection=roi, reducer=ee.Reducer.mean(), scale=10
        ).getInfo()
df = format_values(results)
df.head()

,B2,B3,B4,B8,CC_1,COUNTRY,ENGTYPE_1,GID_0,GID_1,HASC_1,ISO_1,NAME_1,NL_NAME_1,TYPE_1,VARNAME_1
0,892.041324,1032.100549,1018.428051,2217.619864,NA,Singapore,Region,SGP,SGP.1_1,NA,NA,Central,NA,Region,NA
1,1007.188920,1191.552757,1174.162151,2502.056017,NA,Singapore,Region,SGP,SGP.2_1,NA,NA,East,NA,Region,NA
2,644.554584,828.279536,724.837611,2520.811121,NA,Singapore,Region,SGP,SGP.3_1,NA,NA,North,NA,Region,NA
3,695.144399,859.685397,789.526614,2484.610876,NA,Singapore,Region,SGP,SGP.4_1,NA,NA,North-East,NA,Region,NA
4,856.337325,1033.909113,1002.770418,2360.996355,NA,Singapore,Region,SGP,SGP.5_1,NA,NA,West,NA,Region,NA


### 25.4.2. Trích xuất giá trị ảnh theo thời gian

In [ ]:
def extract_values(image):
        # Extract the date from the image
        date = image.date().format("YYYY-MM-dd")
        # Reduce the image by the polygons
        stats = image.reduceRegions(
            collection=roi, reducer=ee.Reducer.mean(), scale=10
        ).map(lambda feature: feature.set("date", date))
        return stats
# Áp dụng hàm extract_values cho toàn bộ ImageCollection
results = sen2col.map(extract_values).flatten().getInfo()
df = format_values(results)
df.head() # Bạn có thể lọc bỏ những cột không cần thiết nếu muốn.

,B2,B3,B4,B8,CC_1,COUNTRY,ENGTYPE_1,GID_0,GID_1,HASC_1,ISO_1,NAME_1,NL_NAME_1,TYPE_1,VARNAME_1,date
0,892.041324,1032.100549,1018.428051,2217.619864,NA,Singapore,Region,SGP,SGP.1_1,NA,NA,Central,NA,Region,NA,2023-07-24
1,1007.188920,1191.552757,1174.162151,2502.056017,NA,Singapore,Region,SGP,SGP.2_1,NA,NA,East,NA,Region,NA,2023-07-24
2,644.554584,828.279536,724.837611,2520.811121,NA,Singapore,Region,SGP,SGP.3_1,NA,NA,North,NA,Region,NA,2023-07-24
3,695.144399,859.685397,789.526614,2484.610876,NA,Singapore,Region,SGP,SGP.4_1,NA,NA,North-East,NA,Region,NA,2023-07-24
4,856.337325,1033.909113,1002.770418,2360.996355,NA,Singapore,Region,SGP,SGP.5_1,NA,NA,West,NA,Region,NA,2023-07-24


## Tóm tắt

Bạn đã hoàn thành Bài 25 và nắm vững kỹ thuật **trích xuất giá trị ảnh theo vị trí** - kỹ năng cốt lõi để kết nối dữ liệu viễn thám với dữ liệu thực địa trên Google Earth Engine.

### Các khái niệm chính đã nắm vững:
- ✅ Trích xuất giá trị pixel tại **dữ liệu điểm** với một ảnh đơn lẻ bằng `sampleRegions()`
- ✅ Trích xuất giá trị theo **chuỗi thời gian** tại nhiều điểm bằng `getRegion()`
- ✅ Tính thống kê vùng (mean, ...) theo **đa giác** với `reduceRegions()`
- ✅ Trích xuất giá trị theo thời gian cho **đa giác** bằng cách kết hợp `map()` + `reduceRegions()` + `flatten()`
- ✅ Chuyển kết quả GEE về **pandas DataFrame** để phân tích và xuất file

### Kỹ năng bạn có thể áp dụng:
- Trích xuất giá trị phổ Sentinel-2 tại các điểm khảo sát thực địa để xây dựng tập dữ liệu huấn luyện
- Tính toán thống kê theo vùng (trung bình, tổng, độ lệch chuẩn) cho nhiều polygon cùng lúc
- Xây dựng time-series giá trị phổ cho từng vùng địa lý phục vụ phân tích xu hướng
- Kết hợp dữ liệu điểm từ GPS với ảnh vệ tinh để kiểm chứng kết quả phân loại
- Xuất kết quả ra Excel/CSV để chia sẻ và phân tích ngoài GEE
